# NB106 - Sprint-2 S2.A: CLRerNet angle-aware LaneIoU

Sprint-1 (NB105) **failed G1**: F1 ceilinged at 0.148 and IoU-aware cls hurt
twice. With Sprint-0 showing the classifier already ranks well (AUC 0.96), the
wall is **geometry**. S2.A replaces the fixed-band LineIoU with CLRerNet's
**LaneIoU** - a slope-scaled IoU band that penalizes near-field shape error
proportional to local angle (our jaggedness zone). Port is unit-tested vs
CLRerNet's own code (`tests/test_lane_iou.py`: ours == ref to 2e-3).

| row | IoU type | where | tests |
|---|---|---|---|
| `s2_baseline`      | line    | -          | **REUSED from NB105 (F1=0.148)** - not re-run |
| `s2_laneiou_loss`  | laneiou | loss only  | does angle-aware loss lift curveIoU off ~0.09? |
| `s2_laneiou_both`  | laneiou | loss+match | + geometry-faithful assignment (full CLRerNet recipe) |

> Only **2 GPU rows** launch; the baseline is reused (loss byte-identical with
> `LANE_IOU_TYPE=line`). tau=0.4 banked on every row so F1 is comparable.

**Gate G2:** F1 >= ~0.45 AND curveIoU >= ~0.25 AND mAP50 >= 0.48 -> promote the
winner to 70k. If curveIoU still saturates ~0.09 -> the anchor/prior paradigm is
the limit -> Sprint 3 (curve-query DETR, MapTR template).


### Cell 1: Mount + deps + locate (mount-alive guarded)

In [2]:
import os, sys, subprocess
from pathlib import Path
os.environ['PYTHONIOENCODING'] = 'utf-8'
REPO_ROOT='/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
MIG=Path(REPO_ROOT)/'stage2/rmt_ppad_migration'
TRAIN=MIG/'P8_train/scripts/train_lane_only.py'
MODEL=MIG/'vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml'
DATA =MIG/'vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only_10k.yaml'
DATASETS=Path('/content/drive/MyDrive/EcoCAR/datasets')
SUBSET=Path('/content/bdd_subset_10k')
def _alive():
    try: return os.path.isdir('/content/drive/MyDrive')
    except OSError: return False
if not _alive():
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=True)
    except Exception as e: print('[mount]', e)
if not _alive():
    raise RuntimeError('Drive mount DEAD (OSError 107). Runtime -> Disconnect '
                       'and delete runtime, reconnect, re-run from Cell 1.')
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
for _p in ('addict','yapf','scipy'):
    try: __import__(_p)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',_p])
try: import mmcv
except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q','mmcv'])
for p in (TRAIN,MODEL,DATA): assert p.exists(), f'missing {p}'
print('[ok] env ready')


Mounted at /content/drive
[ok] env ready


### Cell 2: Extract the 10k subset + point the data YAML at it (idempotent)

In [3]:
PREP=MIG/'extensions/bezier_lcm/scripts/prepare_bdd_subset_10k.py'
req=[DATASETS/'bdd100k_clrkd_curve.tar',
     Path('/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip'),
     DATASETS/'lane_targets_clr_v1_polyline.tar.gz']
miss=[str(p) for p in req if not p.exists()]
if miss: raise FileNotFoundError('Missing:\n  '+'\n  '.join(miss))
if (SUBSET/'prep_summary.json').exists() or (
        (SUBSET/'images/val2017').exists() and any((SUBSET/'images/val2017').iterdir())):
    print('[ok] subset present; skipping')
else:
    cmd=[sys.executable,'-u',str(PREP),'--out-root',str(SUBSET),
         '--n-train','10000','--n-val','2000','--seed','89']
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait()!=0: raise RuntimeError('subset prep failed')
def _yset(y,f,v):
    txt=y.read_text(encoding='utf-8'); line=f'{f}: {v}'
    if line in txt: return
    L=txt.splitlines()
    for i,ln in enumerate(L):
        if ln.strip().startswith(f'{f}:'): L[i]=line; break
    else: L.append(line)
    y.write_text('\n'.join(L)+'\n',encoding='utf-8'); print(f'  [yaml] {f} -> {v}')
def _yunset(y,f):
    txt=y.read_text(encoding='utf-8')
    keep=[ln for ln in txt.splitlines() if not ln.strip().startswith(f'{f}:')]
    n='\n'.join(keep)+'\n'
    if n!=txt: y.write_text(n,encoding='utf-8'); print(f'  [yaml] removed {f}')
_yset(DATA,'path','/content/bdd_subset_10k')
_yset(DATA,'lane_targets_root','/content/bdd_subset_10k/lane_targets')
_yunset(DATA,'drivable_masks_root')
print('[ok] data yaml configured')


[NB89.prep10k] out_root=/content/bdd_subset_10k n_train=10000 n_val=2000
[prep] step 1: extract curve tarball
  extracting bdd100k_clrkd_curve.tar -> /content/bdd_curve_scratch
  done in 108.9s
  curve images: train=/content/bdd_curve_scratch/images/train (70000 jpgs), val=/content/bdd_curve_scratch/images/val (10000 jpgs)
[prep] step 2: hardlink image subset
  sampled 10000 train + 2000 val stems
  on-disk: 10000 train, 2000 val
[prep] step 3: extract matching detection labels
  labels zip: 80003 entries; first 3: ['labels/', 'labels/val2017/', 'labels/val2017/c8b4e0ea-9581068d.txt']
  extracted labels: train=10000 val=2000
[prep] step 4: extract matching lane_targets .pt files
  lane tar first entries: ['lane_targets/train2017/adfc8f5c-8bd2f72d.pt', 'lane_targets/train2017/2beccce2-18444154.pt', 'lane_targets/train2017/97f0a30d-9e0685bf.pt']
  extracted lane targets: train=10000 val=2000

[prep] final inventory:
  images/train2017               10000
  images/val2017                 

### Cell 3: `launch(name, iou_type=...)` helper

NB101 recipe (clrkd, hungarian, clamp=100, fliplr=0.5, wd=0.05) + tau=0.4
banked. Only `--lane-iou-type` varies. Resume-safe; syncs to Drive each epoch.


In [4]:
from stage2.scripts.notebook_utils import run_streaming
PROJECT='/content/runs/sprint2'
LOG_DIR='/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'; os.makedirs(LOG_DIR, exist_ok=True)
EPOCHS,BATCH,LR0,PATIENCE=15,32,'4e-4',99
def launch(name, iou_type='line', iou_match='follow'):
    # iou_type = LaneIoU in the LOSS; iou_match = LaneIoU in the MATCHER
    # ('follow' = same as iou_type). loss-only -> iou_type='laneiou',
    # iou_match='line'; both -> iou_type='laneiou', iou_match='follow'.
    cmd=[sys.executable,'-u',str(TRAIN),
         '--mode','full','--model-yaml',str(MODEL),'--data-yaml',str(DATA),
         '--project',PROJECT,'--name',name,'--device','0',
         '--save-period','15','--batch',str(BATCH),'--epochs',str(EPOCHS),
         '--lr0',LR0,'--patience',str(PATIENCE),
         '--fliplr','0.5','--weight-decay','0.05',
         '--lane-match','hungarian','--lane-weights','clrkd','--diff-clamp','100',
         '--lane-eval-tau','0.4','--lane-iou-type',iou_type,
         '--lane-iou-match',iou_match]
    print(f'\n=== {name}: loss-iou={iou_type} match-iou={iou_match} (tau=0.4) ===\n', flush=True)
    rc=run_streaming(cmd, log_path=os.path.join(LOG_DIR,f'NB106_{name}.log'), check=False)
    print(f'[{name}] rc={rc}  (watch curveIoU, lane_f1, mAP50)')
    return rc==0
print('[ready]')


[ready]


### Cell 4: Row 0: baseline - REUSE NB105 round-1 weights (do NOT re-run)

In [5]:
# Baseline REUSED from NB105 (LANE_IOU_TYPE=line -> loss byte-identical).
# F1=0.148, curveIoU=0.517, mAP50=0.671 @15ep. Aggregator reads it.
print('[baseline] reusing NB105 s1_baseline (F1=0.148) - not re-run')


[baseline] reusing NB105 s1_baseline (F1=0.148) - not re-run


### Cell 5: Row 1: LaneIoU in the regression LOSS ONLY (matcher stays LineIoU)

In [6]:
launch('s2_laneiou_loss', iou_type='laneiou', iou_match='line')


流式输出内容被截断，只能显示最后 5000 行内容。
       1/15        50G      39.75          0      13.91  6.415e-05      94.22     0.5047     0.5015        388        640:  76%|███████▌  | 237/313 [02:29<00:44,  1.73it/s]
       1/15        50G       39.7          0      13.89  6.441e-05      82.55      0.505     0.5015        359        640:  76%|███████▌  | 237/313 [02:30<00:44,  1.73it/s]
       1/15        50G      39.66          0      13.87  6.466e-05      100.3     0.5048     0.5014        319        640:  76%|███████▌  | 237/313 [02:30<00:44,  1.73it/s]
       1/15        50G      39.61          0      13.86  6.492e-05      92.26     0.5053     0.5014        371        640:  76%|███████▌  | 237/313 [02:31<00:44,  1.73it/s]
       1/15        50G      39.56          0      13.84  6.518e-05      89.36      0.505     0.5014        333        640:  76%|███████▌  | 237/313 [02:31<00:44,  1.73it/s]
       1/15        50G      39.51          0      13.82  6.543e-05      95.92     0.5052     0.5013        2

True

### Cell 6: Row 2: LaneIoU in loss AND matcher cost (full CLRerNet recipe)

In [7]:
launch('s2_laneiou_both', iou_type='laneiou', iou_match='laneiou')


流式输出内容被截断，只能显示最后 5000 行内容。
       1/15        50G      39.62          0      13.93  6.415e-05      82.82     0.5045     0.5019        388        640:  76%|███████▌  | 237/313 [02:29<00:44,  1.70it/s]
       1/15        50G      39.57          0      13.91  6.441e-05      79.97     0.5051     0.5019        359        640:  76%|███████▌  | 237/313 [02:29<00:44,  1.70it/s]
       1/15        50G      39.53          0      13.89  6.466e-05      89.07     0.5048     0.5018        319        640:  76%|███████▌  | 237/313 [02:30<00:44,  1.70it/s]
       1/15        50G      39.48          0      13.88  6.492e-05      85.55      0.505     0.5018        371        640:  76%|███████▌  | 237/313 [02:31<00:44,  1.70it/s]
       1/15        50G      39.43          0      13.86  6.518e-05      81.01     0.5048     0.5017        333        640:  76%|███████▌  | 237/313 [02:31<00:44,  1.70it/s]
       1/15        50G      39.38          0      13.84  6.543e-05      82.89     0.5049     0.5019        2

True

### Cell 9: Aggregate - rank by curveIoU + lane_f1, check G2

curveIoU is the key Sprint-2 signal (it was saturated ~0.09 -> LaneIoU should
lift it). G2 = F1>=0.45 AND curveIoU>=0.25 AND mAP50>=0.48.


In [8]:
import csv
from pathlib import Path
PROJECT=Path('/content/runs/sprint2')
DRIVE_CK=Path('/content/drive/MyDrive/EcoCAR/training_runs/checkpoints')
ROWS=['s2_baseline','s2_laneiou_loss','s2_laneiou_both']
_BASE={'metrics/lane_f1(lane)':'0.1483','metrics/lane_curveIoU(lane)':'0.5166',
       'metrics/lane_score_std(lane)':'0.0824','metrics/lane_length_mean(lane)':'0.1052',
       'metrics/mAP50(B)':'0.6712'}
def final(name):
    # s2_baseline reuses NB105's s1_baseline folder.
    cands=[PROJECT/name, DRIVE_CK/name]
    if name=='s2_baseline': cands += [Path('/content/runs/sprint1/s1_baseline'), DRIVE_CK/'s1_baseline']
    for b in cands:
        p=b/'results.csv'
        if p.exists():
            rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(open(p))]
            if rows: return rows[-1]
    if name=='s2_baseline': return _BASE
    return None
def g(r,*ks):
    for k in ks:
        for ck in (r or {}):
            if ck.strip().lower()==k.lower():
                try: return float(r[ck])
                except: pass
    return None
print(f'{"row":18} {"F1":>7} {"curveIoU":>9} {"mAP50":>7} {"len_mean":>9} {"G2":>4}')
print('-'*60)
best=None
for n in ROWS:
    r=final(n)
    if r is None: print(f'{n:18} (not run)'); continue
    f1=g(r,'metrics/lane_f1(lane)'); ci=g(r,'metrics/lane_curveIoU(lane)')
    mp=g(r,'metrics/mAP50(B)'); lm=g(r,'metrics/lane_length_mean(lane)')
    g2=(f1 or 0)>=0.45 and (ci or 0)>=0.25 and (mp or 0)>=0.48
    def fmt(x): return f'{x:.4f}' if isinstance(x,float) else '  -  '
    print(f'{n:18} {fmt(f1):>7} {fmt(ci):>9} {fmt(mp):>7} {fmt(lm):>9} {("Y" if g2 else "-"):>4}')
    if f1 is not None and (mp or 0)>=0.48 and (best is None or f1>best[1]): best=(n,f1,ci)
print()
if best: print(f'BEST (mAP50>=0.48): {best[0]}  F1={best[1]:.4f}  curveIoU={best[2]:.4f}')
print('G2: F1>=0.45 + curveIoU>=0.25 + mAP50>=0.48 -> promote to 70k.')
print('curveIoU rising off ~0.09 = LaneIoU working even if F1 short of 0.45.')
print('If curveIoU still ~0.09 -> anchor paradigm is the wall -> Sprint 3 (curve-query DETR).')


row                     F1  curveIoU   mAP50  len_mean   G2
------------------------------------------------------------
s2_baseline         0.1483    0.5166  0.6712    0.1052    -
s2_laneiou_loss     0.0494    0.5997  0.6796    0.1055    -
s2_laneiou_both     0.2625    0.6098  0.6732    0.1073    -

BEST (mAP50>=0.48): s2_laneiou_both  F1=0.2625  curveIoU=0.6098
G2: F1>=0.45 + curveIoU>=0.25 + mAP50>=0.48 -> promote to 70k.
curveIoU rising off ~0.09 = LaneIoU working even if F1 short of 0.45.
If curveIoU still ~0.09 -> anchor paradigm is the wall -> Sprint 3 (curve-query DETR).
